*The following lab was inspired in the work of Santiago Ríos Guiral (santiago.riosg@udea.edu.co), Santiago Taborda Echeverri (santiago.tabordae@udea.edu.co) and Brayan David Ramos Caicedo (brayan.ramos@udea.edu.co).*

# **Attack Classification**

In the following practice we will analyze the [UNSW-NB15](https://research.unsw.edu.au/projects/unsw-nb15-dataset) database. This database was generated with real and synthetic traffic. The .pcap files are available for its manipulation, however we will use a data set that contains per-flow characteristics of the aforementioned captures. If you want to check in detail how this database was constructed, you can check this [article](https://ieeexplore.ieee.org/abstract/document/7348942).

We will use per-flow generated features (a flow is an unidirectional sequence of packets that have some protocol-header values in common, for example, source and destination IP and ports). This is widely used in network traffic classification, since using per-packet features offer, in general, worse performances compared to its flow-based counterpart. An article explaining the differences between both type of features can be found [here](http://conferences.sigcomm.org/imc/2004/papers/p135-roughan.pdf).  

In [ ]:
%%capture
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1OzTpumhhK3T_juCxLSDe8D7ZuQYWjXhm' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1OzTpumhhK3T_juCxLSDe8D7ZuQYWjXhm" -O UNSW_NB15_testing-set.csv && rm -rf /tmp/cookies.txt
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1I6Tp90l4qVB9Xg3gtIomsDQD2yzeG3te' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1I6Tp90l4qVB9Xg3gtIomsDQD2yzeG3te" -O UNSW_NB15_training-set.csv && rm -rf /tmp/cookies.txt
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1tuXSTha6y41f-hcd4pdZhv1_7EeRrJr_' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1tuXSTha6y41f-hcd4pdZhv1_7EeRrJr_" -O UNSW_NB15_complete-set.csv && rm -rf /tmp/cookies.txt
!wget -nc --no-cache -O init.py -q https://raw.githubusercontent.com/rramosp/2021.deeplearning/main/content/init.py
import init; init.init(force_download=False);

In [ ]:

import numpy as np
import pandas as pd
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import scale

/usr/local/lib/python3.7/dist-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


Let's import the dataset and check how it looks:

In [ ]:
df_attack=pd.read_csv("/content/UNSW_NB15_complete-set.csv")
df_attack.tail()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
257668,82328,0.000005,udp,-,INT,2,0,104,0,200000.005100,254,0,8.320000e+07,0.000000,0,0,0.005000,0.0,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.000000,52,0,0,0,1,2,2,1,1,2,0,0,0,2,1,0,Normal,0
257669,82329,1.106101,tcp,-,FIN,20,8,18062,354,24.410067,254,252,1.241044e+05,2242.109863,7,1,55.880051,143.7,4798.130981,190.980813,255,1072535109,3284291478,255,0.173208,0.100191,0.073017,903,44,0,0,1,1,2,1,1,1,0,0,0,3,2,0,Normal,0
257670,82330,0.000000,arp,-,INT,1,0,46,0,0.000000,0,0,0.000000e+00,0.000000,0,0,60000.720000,0.0,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.000000,46,0,0,0,1,2,1,1,1,1,0,0,0,1,1,1,Normal,0
257671,82331,0.000000,arp,-,INT,1,0,46,0,0.000000,0,0,0.000000e+00,0.000000,0,0,60000.732000,0.0,10.954518,0.000000,0,0,0,0,0.000000,0.000000,0.000000,46,0,0,0,1,2,1,1,1,1,0,0,0,1,1,1,Normal,0
257672,82332,0.000009,udp,-,INT,2,0,104,0,111111.107200,254,0,4.622222e+07,0.000000,0,0,0.009000,0.0,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.000000,52,0,0,0,1,2,1,1,1,1,0,0,0,1,1,0,Normal,0


Try to analize different characteristics of the dataset (size, number of samples per class, number of classes, feature types, measures of central tendency)

In [ ]:
df_attack['attack_cat'].value_counts()

Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174
Name: attack_cat, dtype: int64

In [ ]:
df_attack.dtypes

id                     int64
dur                  float64
proto                 object
service               object
state                 object
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
ct_src_dport_l

In [ ]:
df_attack.shape

(257673, 45)

In [ ]:
df_attack.columns

Index(['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes',
       'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss',
       'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin',
       'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth',
       'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm',
       'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label'],
      dtype='object')

In [ ]:
df_attack.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257673 entries, 0 to 257672
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 257673 non-null  int64  
 1   dur                257673 non-null  float64
 2   proto              257673 non-null  object 
 3   service            257673 non-null  object 
 4   state              257673 non-null  object 
 5   spkts              257673 non-null  int64  
 6   dpkts              257673 non-null  int64  
 7   sbytes             257673 non-null  int64  
 8   dbytes             257673 non-null  int64  
 9   rate               257673 non-null  float64
 10  sttl               257673 non-null  int64  
 11  dttl               257673 non-null  int64  
 12  sload              257673 non-null  float64
 13  dload              257673 non-null  float64
 14  sloss              257673 non-null  int64  
 15  dloss              257673 non-null  int64  
 16  si

In [ ]:
df_attack.describe()

,id,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,label
count,257673.000000,257673.000000,257673.000000,257673.000000,2.576730e+05,2.576730e+05,2.576730e+05,257673.000000,257673.000000,2.576730e+05,2.576730e+05,257673.000000,257673.000000,257673.000000,257673.000000,2.576730e+05,257673.000000,257673.000000,2.576730e+05,2.576730e+05,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,2.576730e+05,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000,257673.000000
mean,72811.823858,1.246715,19.777144,18.514703,8.572952e+03,1.438729e+04,9.125391e+04,180.000931,84.754957,7.060869e+07,6.582143e+05,4.889317,6.743691,912.300834,98.915462,5.419373e+03,582.251456,121.753661,1.006120e+09,1.002295e+09,119.254629,0.046038,0.023652,0.022386,137.639027,121.649703,0.102242,1.968900e+03,9.383176,1.324978,6.050467,5.238271,4.032677,8.322964,0.012819,0.012850,0.132005,6.800045,9.121049,0.014274,0.639077
std,48929.917641,5.974305,135.947152,111.985965,1.737739e+05,1.461993e+05,1.603446e+05,102.488268,112.762131,1.857313e+08,2.412372e+06,65.574953,53.702222,6922.153239,1094.048691,4.903450e+04,3930.153369,127.367443,1.367795e+09,1.363877e+09,127.230477,0.092908,0.053856,0.045771,205.901118,254.041013,0.710593,4.962523e+04,10.829706,0.992300,8.173749,8.160822,5.831515,11.120754,0.116091,0.116421,0.681854,8.396266,10.874752,0.118618,0.480269
min,1.000000,0.000000,1.000000,0.000000,2.400000e+01,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,24.000000,0.000000,0.000000,0.000000e+00,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000
25%,32210.000000,0.000008,2.000000,0.000000,1.140000e+02,0.000000e+00,3.078928e+01,62.000000,0.000000,1.231800e+04,0.000000e+00,0.000000,0.000000,0.008000,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,57.000000,0.000000,0.000000,0.000000e+00,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,2.000000,2.000000,0.000000,0.000000
50%,64419.000000,0.004285,4.000000,2.000000,5.280000e+02,1.780000e+02,2.955665e+03,254.000000,29.000000,7.439423e+05,1.747441e+03,0.000000,0.000000,0.381696,0.007000,6.736370e-01,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,73.000000,44.000000,0.000000,0.000000e+00,5.000000,1.000000,2.000000,1.000000,1.000000,3.000000,0.000000,0.000000,0.000000,3.000000,4.000000,0.000000,1.000000
75%,110923.000000,0.685777,12.000000,10.000000,1.362000e+03,1.064000e+03,1.250000e+05,254.000000,252.000000,8.000000e+07,2.210538e+04,3.000000,2.000000,58.094727,56.438859,2.787367e+03,119.712937,255.000000,2.007375e+09,1.992752e+09,255.000000,0.082082,0.036842,0.044665,100.000000,89.000000,0.000000,0.000000e+00,12.000000,2.000000,6.000000,4.000000,3.000000,8.000000,0.000000,0.000000,0.000000,8.000000,11.000000,0.000000,1.000000
max,175341.000000,59.999989,10646.000000,11018.000000,1.435577e+07,1.465753e+07,1.000000e+06,255.000000,254.000000,5.988000e+09,2.242273e+07,5319.000000,5507.000000,84371.496000,57739.240000,1.483831e+06,463199.240100,255.000000,4.294959e+09,4.294882e+09,255.000000,3.821465,3.226788,2.928778,1504.000000,1500.000000,172.000000,6.558056e+06,63.000000,6.000000,59.000000,59.000000,46.000000,65.000000,4.000000,4.000000,30.000000,60.000000,62.000000,1.000000,1.000000


A good practice is to check how disperse are the features, analyze its skewness and identify outliers. Boxplots are good tools to check this, try to select different features and plot them. Since we are dealing with different types of attacks, is recommended that you generate a boxplot per each feature related to each label. For example, if you want to analyze the feature *smean*, you should draw 10 boxplots (9 for all the attack types and 1 for the normal traffic). Draw your own conclusions.



In [ ]:
training_DB = pd.read_csv("/content/UNSW_NB15_complete-set.csv", index_col=0)
df_analysis = pd.DataFrame({'dload':training_DB['dload'][:] , 'label': training_DB['label'][:]})
protos = list(training_DB['proto'].unique())
protos_dict = {}

for index, proto in enumerate (protos):
    protos_dict[proto] = index

training_DB_proto = training_DB.copy()
training_DB_proto = training_DB_proto.replace(protos_dict)
training_DB_proto['proto'].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132])

We recommend you to check the following features, however, you are free to choose and compare the variables you consider.

In [ ]:
features1 = ['sttl','ct_dst_src_ltm','spkts','dload','sloss','dloss','ct_src_ltm','ct_srv_dst']
features2 = ['sbytes','sttl','smean','ct_dst_sport_ltm']
significant_features = ['proto','sttl','dmean','ct_dst_src_ltm','ct_dst_sport_ltm','rate','dttl','dload','dtcpb','stcpb']
feat = ['sttl','dmean','ct_dst_src_ltm','ct_dst_sport_ltm','rate','dttl','dload','dtcpb','stcpb']

#Put your code in order to graph the distributions of the variables

We will try to analyze three different problems. First, we will try to solve a two-class problem, aiming to classify normal traffic and anomalous traffic. Here is important to have in mind that the samples that are labeled as "attack" come from different types of attacks (worms, DoS, Fuzzers, among others). Then we will pick a particular attack (e.g. DDoS) and try to differentiate it from normal traffic. Finally, we will solve a multi-class problem, where we will try to simultaneously classify several attacks.

Sometimes models trained on unbalanced datasets offer poor results when used to classify unseen samples. This is because some algorithms, when receiving more data from one particular class, could be biased towards that particular group of observations, failing to understand the patterns used to distinguish among classes and leading to overfitting the bigger class. Since there is more data from the majority class, the [accuracy paradox](https://medium.com/strands-tech-corner/unbalanced-datasets-what-to-do-144e0552d9cd) can be present, since you could get a model that assign the same class to all observations.

There are different alternatives to deal with this issue, in this case we will use undersampling, trying to reduce the ratio between classes, just selecting random observations without replacement. If you want to do an "informed" undersampling, you could use a clustering technique first to retain the most of the underlying information of each class.


In [ ]:
df_attack['attack_cat'].value_counts()

Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174
Name: attack_cat, dtype: int64

In [ ]:
df_attack['label'].value_counts()

1    164673
0     93000
Name: label, dtype: int64

The dataset contains some categorical variables, these features can be coded and used in some particular algorithms, if you want, you can drop these columns and solve the classification problem. Further work could be trying to codify these features and choosing an algorithm that fits.

In [ ]:
df_attack_ub = df_attack.drop(columns=['id','proto','service','state'])
df_attack_ub.dtypes


dur                  float64
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
ct_src_dport_ltm       int64
ct_dst_sport_ltm       int64
ct_dst_src_ltm         int64
is_ftp_login           int64
ct_ftp_cmd    

In [ ]:
df_attack_ub.head()

,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.121478,6,4,258,172,74.087490,252,254,14158.942380,8495.365234,0,0,24.295600,8.375000,30.177547,11.830604,255,621772692,2202533631,255,0.000000,0.000000,0.000000,43,43,0,0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0
1,0.649902,14,38,734,42014,78.473372,62,252,8395.112305,503571.312500,2,17,49.915000,15.432865,61.426934,1387.778330,255,1417884146,3077387971,255,0.000000,0.000000,0.000000,52,1106,0,0,43,1,1,1,1,2,0,0,0,1,6,0,Normal,0
2,1.623129,8,16,364,13186,14.170161,62,252,1572.271851,60929.230470,1,6,231.875571,102.737203,17179.586860,11420.926230,255,2116150707,2963114973,255,0.111897,0.061458,0.050439,46,824,0,0,7,1,2,1,1,3,0,0,0,2,6,0,Normal,0
3,1.681642,12,12,628,770,13.677108,62,252,2740.178955,3358.622070,1,3,152.876547,90.235726,259.080172,4991.784669,255,1107119177,1047442890,255,0.000000,0.000000,0.000000,52,64,0,0,1,1,2,1,1,3,1,1,0,2,1,0,Normal,0
4,0.449454,10,6,534,268,33.373826,254,252,8561.499023,3987.059814,2,1,47.750333,75.659602,2415.837634,115.807000,255,2436137549,1977154190,255,0.128381,0.071147,0.057234,53,45,0,0,43,1,2,2,1,40,0,0,0,2,39,0,Normal,0


First, try to solve the two class classification problem between normal traffic (labeled as "0"), and attack samples (labeled as "1"). Here you can try to solve the problem with an unbalanced dataset and then check if there is any improvement working with a balanced dataset. We recommend you to use a 70-30 train-test split. Try to use at least two types of classifiers that can handle continuos and discrete features.

**2 Classes (normal and attack) - Unbalanced**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.model_selection import cross_val_score


#Put your logic to split the data and implement the algorithm
#Use accuracy metrics and depict the confusion matrix especially in multiclass problems

**2 Classes (normal and attack) - balanced**

In [ ]:
#We suggest you to use the following methods to undersample the data. You are also free
#to use another undersampling method (e.g. Using a clustering method)
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
under_sampler = RandomUnderSampler(random_state=42)
x_res, y_res = under_sampler.fit_resample(x_train, y_train)
x_res_test, y_res_test = under_sampler.fit_resample(x_test, y_test)

**2 Classes (normal and DDoS) - unbalanced**

In [ ]:
df2_ub = df_attack_ub.loc[(df_attack_ub['attack_cat']=='Normal') | (df_attack_ub['attack_cat']=='DoS')]
df2_ub['attack_cat'].value_counts()

Normal    93000
DoS       16353
Name: attack_cat, dtype: int64

In [ ]:
df2_ub.head()

,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.121478,6,4,258,172,74.087490,252,254,14158.942380,8495.365234,0,0,24.295600,8.375000,30.177547,11.830604,255,621772692,2202533631,255,0.000000,0.000000,0.000000,43,43,0,0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0
1,0.649902,14,38,734,42014,78.473372,62,252,8395.112305,503571.312500,2,17,49.915000,15.432865,61.426934,1387.778330,255,1417884146,3077387971,255,0.000000,0.000000,0.000000,52,1106,0,0,43,1,1,1,1,2,0,0,0,1,6,0,Normal,0
2,1.623129,8,16,364,13186,14.170161,62,252,1572.271851,60929.230470,1,6,231.875571,102.737203,17179.586860,11420.926230,255,2116150707,2963114973,255,0.111897,0.061458,0.050439,46,824,0,0,7,1,2,1,1,3,0,0,0,2,6,0,Normal,0
3,1.681642,12,12,628,770,13.677108,62,252,2740.178955,3358.622070,1,3,152.876547,90.235726,259.080172,4991.784669,255,1107119177,1047442890,255,0.000000,0.000000,0.000000,52,64,0,0,1,1,2,1,1,3,1,1,0,2,1,0,Normal,0
4,0.449454,10,6,534,268,33.373826,254,252,8561.499023,3987.059814,2,1,47.750333,75.659602,2415.837634,115.807000,255,2436137549,1977154190,255,0.128381,0.071147,0.057234,53,45,0,0,43,1,2,2,1,40,0,0,0,2,39,0,Normal,0


**2 Classes (normal and DDoS) - balanced**




**Multiclass problem - unbalanced**

In [ ]:
df_attack_ub = df_attack.drop(columns=['id','proto','service','state','label'])
att = {'Backdoor':0, 'Analysis':1, 'Fuzzers':2, 'Shellcode':3,'Reconnaissance':4, 'Exploits':5, 'DoS':6, 'Worms':7,'Generic':8, 'Normal':9}
df3_ub = df_attack_ub
df3_ub.replace({'attack_cat':att},inplace=True)

In [ ]:
df3_ub.head()

**Multiclass problem - balanced**

**Multiclass problem - balanced - removing classes with little data**

In [ ]:
df3_ub['attack_cat'].value_counts()

In [ ]:
df_attack_ub = df_attack.drop(columns=['id','proto','service','state','label'])
df4_ub = df_attack_ub
df4_ub = df4_ub.loc[(df4_ub['attack_cat']!='Worms') & (df4_ub['attack_cat']!='Shellcode')& (df4_ub['attack_cat']!='Backdoor')& (df4_ub['attack_cat']!='Analysis')]

att = {'Backdoor':0, 'Analysis':1, 'Fuzzers':2, 'Shellcode':3,'Reconnaissance':4, 'Exploits':5, 'DoS':6, 'Worms':7,'Generic':8, 'Normal':9}
df4_ub.replace({'attack_cat':att},inplace=True)
df4_ub['attack_cat'].value_counts()

Now, pick one of the previous problems and check the results choosing the features that you consider provide the most information. Check if there is any improvement in the accuracy.



Finally read about data standarization. Check if you can apply it to any of the features. Again, draw your own conclusions.